# LinkedIn Jobs Scraper Việt Nam (IT) - Phiên bản Jupyter Notebook

Script này scrape job IT từ LinkedIn Việt Nam sử dụng Voyager API unofficial.

**Cảnh báo**:
- Đây là unofficial API → LinkedIn có thể block IP/account bất cứ lúc nào.
- Phải cập nhật `csrf-token` và `li_at` cookie thường xuyên (lấy từ DevTools khi login LinkedIn).
- Chạy chậm, có sleep để tránh block.
- Năm 2026: Endpoint có thể thay đổi → kiểm tra Network tab trên browser nếu lỗi.

Chạy từng cell theo thứ tự.

In [1]:
# Cell 1: Import thư viện
import requests
import json
from urllib.parse import quote
import csv
import time
import datetime
import os
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed

print("Import thành công!")

Import thành công!


c:\Users\trand\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
# Cell 2: Cấu hình chính (CHỈNH Ở ĐÂY)

SUMMARY_FILE = "linkedin_job_summaries.jsonl"
CSV_FILE = "linkedin_jobs_vietnam_it_full.csv"
COUNT = 100
MAX_WORKERS = 8
RETRY_COUNT = 3

load_dotenv()

# Headers từ curl mới nhất
headers = {
    "accept": "application/vnd.linkedin.normalized+json+2.1",
    "accept-language": "en-US,en;q=0.9,vi;q=0.8",
    "csrf-token": "ajax:1656999207179305697",
    "priority": "u=1, i",
    "referer": "https://www.linkedin.com/jobs/search/",
    "sec-ch-prefers-color-scheme": "light",
    "sec-ch-ua": '"Not:A-Brand";v="99", "Google Chrome";v="145", "Chromium";v="145"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-origin",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
    "x-li-lang": "vi_VN",
    "x-li-page-instance": "urn:li:page:d_flagship3_search_srp_jobs;77AFEL7YRiWr2ez/0uP6Vw==",
    "x-li-pem-metadata": "Voyager - Careers - Jobs Search=jobs-search-results,Voyager - Careers - Critical - careers-api=jobs-search-results",
    "x-li-prefetch": "1",
    "x-li-track": '{"clientVersion":"1.13.42665","mpVersion":"1.13.42665","osName":"web","timezoneOffset":7,"timezone":"Asia/Saigon","deviceFormFactor":"DESKTOP","mpName":"voyager-web","displayDensity":1.125,"displayWidth":1728,"displayHeight":972}',
    "x-restli-protocol-version": "2.0.0",
}

# ⚠️ QUAN TRỌNG: Tất cả cookies từ curl -b flag
# Cần cập nhật thường xuyên khi LinkedIn yêu cầu!
cookies = {
    "bscookie": 'v=1&202511160954570107781a-c6b2-4727-83a6-e6848d03d807AQHdfzylSvpdVMcNABUonxDPCUCo0zQD',
    "g_state": '{"i_l":0}',
    "li_rm": "AQF--9-49_8nvQAAAZqibTHHJFguEjlWFPJfylXip-C-Ntz7evPei6zRvIzAkf0SaZbTzJcmYwr6WtY5EbYqd0A0AThPtQSuUQjetyTrUAtZfpzkk3ki7d5Y",
    "li_theme": "light",
    "li_theme_set": "app",
    "li_sugr": "96e59326-2816-4f83-95aa-052a94c3b490",
    "bcookie": 'v=2&67d253d6-a6e8-4765-8747-9ead163cf620',
    "timezone": "Asia/Saigon",
    "AMCVS_14215E3D5995C57C0A495C55@AdobeOrg": "1",
    "dfpfpt": "b3ebf4bcaca245e190e51ca42319199d",
    "_pxvid": "23010cfb-1397-11f1-950c-7f6137f5a050",
    "visit": "v=1&M",
    "sdui_ver": "sdui-flagship:0.1.30166+SduiFlagship0",
    "fptctx2": "taBcrIH61PuCVH7eNCyH0HyAAKgSb15ZEqidLg30r8MKpWVTBj4P3iqkYmR07O5r7JnYfvJLoSoaSOnT9TqwHIAHALHCO+NCIUACk18y8fHT6vg44GIfdFzUptVvcFDkjKv4Jf0M2vqdblJGjkGmYIrJKBQhVNMCUqOgIclU2DVb1Gqzym0ibsj6SBDHJPVY04GeWhJX7qEVO2H1Bo0E8x5vluJVOeGjt+Eveee59xpnyger+0pIGBbAG5Zq5xIEwvbxAaHz9i5B/065mAE4uX4Wu+E5m6UAQChstKpphyBP5GwOTrXrfOyMNs/hFAe/Hr75w6rAuOPtaes3Xf/FhipWMk52l+6FpPHo+ZAUYCA=",
    "liap": "true",
    "li_at": "AQEDAV_UGWUDwqUEAAABnMaYUwEAAAGc6qTXAVYADCrjsXVNBD96Nt593qY0UpzS8I-1HmLYRWUtWaPC0eUV947VbKHlALiQfJVkDow5w3c54je12eRg3Onlx_AW9RjLhxWOs7t6lpsXYRF0LxgPV5ey",
    "JSESSIONID": '"ajax:1656999207179305697"',
    "lang": "v=2&lang=vi-vn",
    "lidc": '"b=OB53:s=O:r=O:a=O:p=O:g=4186:u=10:x=1:i=1772860700:t=1772944554:v=2:sig=AQFh73TUKSzvoqanO_pApW_HjjX77AT7"',
    "UserMatchHistory": "AQKPQ_4gRIYIaAAAAZzGu341_XaBr9ovSIyGg5TXtWo7EV9kOfBbvtFUAEeSlPXJ3-uTS7iL9_8smI5mmJPGhL8KPgex6E_fGe9R1wOGhdNCXx3C5SYnkU-zej-C1ZHOGr0ADwC71D-4uxLrb6z1tDAPiLr6mvA1Kfdu7XB746A6mEEmhvQem46ElvlEhxkLcI56gXlvA9Stf-b-5Ke3weDMl9hn0HglU_D7ZN1rQFx5GGp9MjFQMt0Mu8MV2-1dynhGHUvqpIsjnQaVeEEaURopOZZydcB9w4O_EVNtkRfwS94tPsUWXdcIIbV3vpLPPOyxrXO_eku9k9t_CGCg",
    "__cf_bm": "TpOPTSvKUCnGLH8eGwyfYw26ohJ0FEeUBpMg06l6z0k-1772861397-1.0.1.1-LZ6.ZuU.AsylN2Rxb87cVWN2f59bE.wxoI96yRJdKniP9yprVClePqMSO1aI_CrVNYvhOTFVeiMAFEAnRYsGGWBcmwQbq3RrijvyL_W6pKM",
    "AMCV_14215E3D5995C57C0A495C55@AdobeOrg": "-637568504|MCIDTS|20520|MCMID|27158816828677600414853319144040392812|MCOPTOUT-1772868665s|NONE|vVersion|5.1.1",
    "_uetsid": "cf3a840019df11f1a714ddddc6f4f495",
    "_uetvid": "7b63d330139811f1bfd21357de95d556",
}

session = requests.Session()
session.headers.update(headers)
session.cookies.update(cookies)

# Test kết nối
test_url = "https://www.linkedin.com/voyager/api/voyagerJobsDashJobCards?decorationId=com.linkedin.voyager.dash.deco.jobs.search.JobSearchCardsCollection-220&count=1&q=jobSearch&query=(origin:JOBS_HOME_KEYWORD_HISTORY,keywords:IT,locationUnion:(geoId:104195383),selectedFilters:(distance:List(25.0)),spellCorrectionEnabled:true)&start=0"

print("=" * 60)
print("TEST KẾT NỐI LINKEDIN (All Cookies)")
print("=" * 60)
print(f"✓ Headers: {len(headers)} items")
print(f"✓ Cookies: {len(cookies)} items")
print()

try:
    print("Gửi request...")
    response = session.get(test_url, timeout=15, allow_redirects=False)
    print(f"Status Code: {response.status_code}")
    
    if response.status_code == 200:
        try:
            data = response.json()
            if 'data' in data:
                total_jobs = data.get('data', {}).get('paging', {}).get('total', 0)
                print(f"✓ Kết nối thành công!")
                print(f"  Included items: {len(data.get('included', []))}")
                print(f"  Tổng jobs tìm được: {total_jobs}")
            else:
                print("✗ Response không có 'data' field")
        except Exception as e:
            print(f"✗ Response không phải JSON: {e}")
    elif response.status_code in [301, 302, 303, 307, 308]:
        print(f"✗ REDIRECT {response.status_code}")
        print("  → Cookies chưa đủ hoặc hết hạn!")
    elif response.status_code in [401, 403]:
        print(f"✗ HTTP {response.status_code} - Unauthorized")
    else:
        print(f"✗ HTTP {response.status_code}")
        
except Exception as e:
    print(f"✗ Lỗi: {e}")

print("=" * 60)

TEST KẾT NỐI LINKEDIN (All Cookies)
✓ Headers: 19 items
✓ Cookies: 24 items

Gửi request...
Status Code: 200
✓ Kết nối thành công!
  Included items: 8
  Tổng jobs tìm được: 1037


In [3]:
# Cell 3: Load keywords từ file (hoặc mặc định)
def load_keywords(filename="keywords.txt"):
    try:
        with open(filename, "r", encoding="utf-8") as f:
            keywords = [line.strip() for line in f if line.strip()]
        print(f"Đã tải {len(keywords)} từ khóa: {keywords}")
        return keywords
    except FileNotFoundError:
        print("Không tìm thấy 'keywords.txt' → dùng mặc định")
        return ["it", "software engineer", "data analyst", "developer vietnam"]

keywords_list = load_keywords()
print("\nDanh sách keywords:", keywords_list)

Đã tải 229 từ khóa: ['IT', 'Information Technology', 'Công nghệ thông tin', 'CNTT', 'Computer Science', 'CS', 'Software', 'Hardware', 'Firmware', 'System', 'Application', 'Platform', 'Infrastructure', 'Technology', 'Tech', 'Lập trình', 'Lập trình viên', 'Programmer', 'Developer', 'Software Developer', 'Software Engineer', 'Kỹ sư phần mềm', 'Frontend Developer', 'Backend Developer', 'Fullstack Developer', 'Web Developer', 'Mobile Developer', 'Senior Developer', 'Senior Engineer', 'Lead Developer', 'Junior Developer', 'Junior Engineer', 'Fresher IT', 'Intern IT', 'Thực tập sinh IT', 'Entry Level', 'Mid Level', 'Remote IT', 'Hybrid IT', 'Onsite', 'Offshore', 'Nearshore', 'Outsourcing', 'ODC', 'Python', 'Java', 'JavaScript', 'TypeScript', 'PHP', '.NET', 'C#', 'Golang', 'Go', 'Ruby', 'Kotlin', 'Swift', 'C', 'C++', 'Rust', 'Scala', 'Objective-C', 'Shell Script', 'Bash', 'PowerShell', 'SQL', 'NoSQL', 'Database', 'DB', 'RDBMS', 'MySQL', 'PostgreSQL', 'Oracle', 'SQL Server', 'MongoDB', 'Redis',

In [4]:
# Cell 4: Load job đã xử lý trước đó (tránh duplicate)
def load_existing_summaries():
    if not os.path.exists(SUMMARY_FILE):
        return set()
    existing_ids = set()
    with open(SUMMARY_FILE, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if (i + 1) % 500 == 0:
                print(f"   Loading summaries cũ: {i+1} dòng...")
            if line.strip():
                try:
                    entry = json.loads(line)
                    existing_ids.add(entry['job_id'])
                except:
                    pass
    print(f"Đã load {len(existing_ids)} job đã xử lý trước đó.")
    return existing_ids

processed_job_ids = load_existing_summaries()
seen_job_ids = processed_job_ids.copy()

   Loading summaries cũ: 500 dòng...
   Loading summaries cũ: 1000 dòng...
   Loading summaries cũ: 1500 dòng...
   Loading summaries cũ: 2000 dòng...
   Loading summaries cũ: 2500 dòng...
   Loading summaries cũ: 3000 dòng...
   Loading summaries cũ: 3500 dòng...
   Loading summaries cũ: 4000 dòng...
   Loading summaries cũ: 4500 dòng...
   Loading summaries cũ: 5000 dòng...
   Loading summaries cũ: 5500 dòng...
   Loading summaries cũ: 6000 dòng...
   Loading summaries cũ: 6500 dòng...
   Loading summaries cũ: 7000 dòng...
   Loading summaries cũ: 7500 dòng...
   Loading summaries cũ: 8000 dòng...
   Loading summaries cũ: 8500 dòng...
   Loading summaries cũ: 9000 dòng...
   Loading summaries cũ: 9500 dòng...
   Loading summaries cũ: 10000 dòng...
   Loading summaries cũ: 10500 dòng...
   Loading summaries cũ: 11000 dòng...
   Loading summaries cũ: 11500 dòng...
   Loading summaries cũ: 12000 dòng...
   Loading summaries cũ: 12500 dòng...
   Loading summaries cũ: 13000 dòng...
   Loa

In [5]:
# Cell 5: Hàm fetch danh sách job (pagination)
base_url = "https://www.linkedin.com/voyager/api/voyagerJobsDashJobCards"

def fetch_job_list(keyword, seen_job_ids):
    query_string_base = (
        f"decorationId=com.linkedin.voyager.dash.deco.jobs.search.JobSearchCardsCollection-220"
        f"&count={COUNT}"
        f"&q=jobSearch"
        f"&query=(origin:JOBS_HOME_KEYWORD_HISTORY,keywords:{keyword},locationUnion:(geoId:104195383),selectedFilters:(distance:List(25.0)),spellCorrectionEnabled:true)"
        f"&start="
    )

    new_jobs = []
    start = 0
    total = None
    page = 1

    print(f"[{keyword.upper()}] Bắt đầu tìm kiếm...")

    while total is None or start < total:
        print(f"   Trang {page} (start={start})...", end=" ")
        url = f"{base_url}?{query_string_base}{start}"
        try:
            response = session.get(url, timeout=30, allow_redirects=False)
            if response.status_code != 200:
                print(f"HTTP {response.status_code}")
                break
            data = response.json()
        except Exception as e:
            print(f"Lỗi: {e}")
            break

        included = data.get('included', [])
        if total is None:
            total = data.get('data', {}).get('paging', {}).get('total', 0)
            print(f"→ Tổng: {total}")

        jobs_found = 0
        for item in included:
            if (item.get('$type') == 'com.linkedin.voyager.dash.jobs.JobPosting' and
                item.get('entityUrn', '').startswith('urn:li:fsd_jobPosting:')):
                job_id = item['entityUrn'].split(':')[-1]
                if job_id not in seen_job_ids:
                    seen_job_ids.add(job_id)
                    new_jobs.append(item)
                    jobs_found += 1

        print(f"+{jobs_found} job mới")
        if jobs_found == 0 and start > 0:
            break

        start += COUNT
        page += 1
        time.sleep(1)

    print(f"[{keyword.upper()}] Hoàn thành: +{len(new_jobs)} job mới\n")
    return new_jobs

In [6]:
# Cell 6: Hàm fetch chi tiết job (có retry)
def fetch_job_detail(job_id):
    url_temp = "https://www.linkedin.com/voyager/api/graphql"
    urn_raw = f"urn:li:fsd_jobPosting:{job_id}"
    urn_encoded = quote(urn_raw, safe="")
    variables = f"(jobPostingUrn:{urn_encoded})"
    query_id = "voyagerJobsDashJobPostings.891aed7916d7453a37e4bbf5f1f60de4"
    url = f"{url_temp}?variables={variables}&queryId={query_id}"

    for attempt in range(RETRY_COUNT):
        try:
            response = session.get(url, timeout=20, allow_redirects=False)
            if response.status_code != 200:
                if attempt == RETRY_COUNT - 1:
                    return None, None
                time.sleep(1)
                continue
            data = response.json()
            included = data.get('included', [])
            for item in included:
                if item.get('$type') == 'com.linkedin.voyager.dash.jobs.JobPosting':
                    return item, included
            return None, None
        except Exception:
            if attempt == RETRY_COUNT - 1:
                return None, None
            time.sleep(2 ** attempt)
    return None, None

In [7]:
# Cell 7: Hàm extract thông tin job
def extract_job_info(job_detail, included=None):
    if not job_detail:
        return None

    def resolve_urn(urn, target_type):
        if not urn or not included:
            return None
        for item in included:
            if item.get('entityUrn') == urn and item.get('$type') == target_type:
                return item
        return None

    job_id = job_detail.get('entityUrn', '').split(':')[-1]
    job_title = job_detail.get('title', '')
    company_name = job_detail.get('companyDetails', {}).get('name')

    location = None
    loc_urn = job_detail.get('*location')
    if loc_urn and isinstance(loc_urn, str):
        geo = resolve_urn(loc_urn, 'com.linkedin.voyager.dash.common.Geo')
        if geo:
            location = geo.get('defaultLocalizedName') or geo.get('abbreviatedLocalizedName')
    if not location:
        location = job_detail.get('formattedLocation')

    description = job_detail.get('description', {}).get('text', '')

    posting_date = None
    if job_detail.get('listedAt'):
        try:
            posting_date = datetime.datetime.fromtimestamp(job_detail['listedAt'] / 1000).strftime('%Y-%m-%d')
        except:
            pass

    # Các field khác tương tự...
    job_url = f"https://www.linkedin.com/jobs/view/{job_id}"

    return {
        'job_id': job_id,
        'job_title': job_title,
        'company': company_name,
        'location': location,
        'description': description,
        'posting_date': posting_date,
        'job_url': job_url,
        'crawled_at': datetime.datetime.now().isoformat(),
        # Thêm các field khác nếu cần
    }

In [8]:
# Cell 8: CHẠY SCRAPING CHÍNH (chạy cell này cuối cùng)

all_new_jobs = []

for idx, kw in enumerate(keywords_list, 1):
    print(f"[{idx}/{len(keywords_list)}] Từ khóa: '{kw}'")
    new_jobs = fetch_job_list(kw, seen_job_ids)
    all_new_jobs.extend(new_jobs)

if not all_new_jobs:
    print("Không có job mới nào. Kết thúc.")
else:
    print(f"\nTổng {len(all_new_jobs)} job mới cần lấy detail.\n")

    file_exists = os.path.exists(CSV_FILE)
    with open(CSV_FILE, 'a', newline='', encoding='utf-8-sig') as csvfile:
        fieldnames = ['job_id', 'job_title', 'company', 'location', 'description',
                      'posting_date', 'job_url', 'crawled_at']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        if not file_exists or os.stat(CSV_FILE).st_size == 0:
            writer.writeheader()

        success = 0
        failed = 0

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(fetch_job_detail, job['entityUrn'].split(':')[-1]): job for job in all_new_jobs}

            processed = 0
            total = len(futures)
            for future in as_completed(futures):
                processed += 1
                job = futures[future]
                job_id = job['entityUrn'].split(':')[-1]
                title = job.get('title', 'No title')

                job_detail, included = future.result()
                source = job_detail or job
                info = extract_job_info(source, included=included)

                if info:
                    writer.writerow(info)
                    csvfile.flush()

                    # Lưu summary
                    entry = {
                        "job_id": job_id,
                        "title": job.get('title'),
                        "entityUrn": job['entityUrn'],
                        "crawled_at": datetime.datetime.now().isoformat(),
                    }
                    with open(SUMMARY_FILE, "a", encoding="utf-8") as f:
                        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

                    success += 1
                    print(f"[{processed}/{total}] [OK] {job_id} - {title}")
                else:
                    failed += 1
                    print(f"[{processed}/{total}] [FAIL] {job_id} - {title}")

                time.sleep(0.25)

    print("\nHOÀN THÀNH!")
    print(f"Thành công: {success} | Fail: {failed} | File: {CSV_FILE}")

[1/229] Từ khóa: 'IT'
[IT] Bắt đầu tìm kiếm...
   Trang 1 (start=0)... → Tổng: 1037
+54 job mới
   Trang 2 (start=100)... +50 job mới
   Trang 3 (start=200)... +64 job mới
   Trang 4 (start=300)... +67 job mới
   Trang 5 (start=400)... +70 job mới
   Trang 6 (start=500)... +66 job mới
   Trang 7 (start=600)... +65 job mới
   Trang 8 (start=700)... +61 job mới
   Trang 9 (start=800)... +60 job mới
   Trang 10 (start=900)... +48 job mới
   Trang 11 (start=1000)... +0 job mới
[IT] Hoàn thành: +605 job mới

[2/229] Từ khóa: 'Information Technology'
[INFORMATION TECHNOLOGY] Bắt đầu tìm kiếm...
   Trang 1 (start=0)... → Tổng: 210
+14 job mới
   Trang 2 (start=100)... +17 job mới
   Trang 3 (start=200)... 

KeyboardInterrupt: 